# **Expectation Maximization Algorithm and Mixture of Gaussians**

En esta sección vamos a cambiar el foco de nuestro problema de manera importante. Hasta ahora estabamos tratando algoritmos de aprendizaje supervisado. Esto quiere decir que nuestro dataset de entrenamiento contenía ejemplos de entrenamientos y los labels, las soluciones. Los algoritmos que trataremos en esta sección son no supervisados que quiere decir que el dataset se nos da sin labels, sin clasificar. Por lo tanto se nos dará un conjunto de ejemplos y debemos encontrar cierto patrón en ellos. 

En concreto nos centraremos en problemas de clustering. Se dos dará una serie de puntos en un espacio, deberemos asumir que que existen unas ciertas clases de forma que cada punto pertenece a una clase y nuestra tarea es asignar cada punto a su clase.

## Expectation Maximization

Como hemos dicho partimos de un dataset $X = \{x_i\}_{i=1}^{i=n}$ que asumimos que es una muestra tomada de una poblacion que se distribuye de acuerdo a una función desconocida. Nuestro objetivo final es estimar esa función de probabilidad $p(x ; \theta)$ que nos dice la probabilidad de que al tomar otra muestra cualquiera sea $x$ donde la funcion esta parametrizada con unos parametros $\theta$.
Procediendo de una forma que cobrara mucho sentido mas adelante vamos a asumir que existen otra variables $z$ que son ocultas a las que llamaremos  latentes. Vamos a tratar de estimar $p(x,z ; \theta)$ de forma que $p(x ; \theta) = \sum\limits_z{p(x,z ; \theta)}$.

Lo que haremos sera tratar de maximizar la verosimilitud de la muestra de acuerdo a los parametros. Es decir definimos $l(\theta)$ una funcion de los parametros que mide como de probable es, dados esos parametros, obtener la muestra $X$.
$$
\begin{align}
l(\theta) &= \sum\limits_{i=1}^np(x^{(i)};\theta)= \\
          &= \sum\limits_{i=1}^n\sum\limits_zp(x^{(i)},z;\theta)
\end{align}
$$
Como el objetivo es maximizar el valor podemos aplicar una funcion monotona a cada lado de la igualdad y el maximo no cambia. Ademas abusando de notacion volveremos a llamar a la funcion resultante $l$.
$$
\begin{align}
l(\theta) &= \sum\limits_{i=1}^n\log p(x^{(i)};\theta)= \\
          &= \sum\limits_{i=1}^n\log \sum\limits_zp(x^{(i)},z;\theta)
\end{align}
$$

Optimizar esta función directamente es un problema de optimización no convexa muy complicado en general pero resulta que si las variables latentes fueran observadas sería mucho más sencillo. 

Además, veremos que podemos construir una cota mínima para la verosimilitud que dependerá de la distribución de la variable latente $z$.

El esquema EM será un esquema de algoritmos iterativos que, en estas condiciones, tratará de maximizar la verosimilitud iterando entre un paso en el que se asumen $\theta$ fijos y se calcula una distribución de $z$ de forma que haga máxima la cota y luego un paso en el que, con $z$ fijos se eligen los parametros de forma que maximicen la verosimilutud, que como $z$ son conocidos será posible. 

Desarrollemos con más cuidado y detalle este algoritmo:

### **1. Cota minima para la verosimilitud**

Por simplificar el desarrollo haremos la derivacion del algoritmo para un solo ejemplo de entrenamiento. Además en este paso usaremos el siguiente teorema.

<u>**Teorema (Desigualdad de Jensen):**</u> Sea $f$ una función convexa y $X$ una variable aleatoria, entonces:
$$
\mathbb{E}[f(X)] \ge f(\mathbb{E}[X])
$$
Además si $f$ es estrictamente convexa la igualdad se da si y solo si $X=\mathbb{E}[X]$ con probabilidad $1$, es decir, es una constante

Con esto en mente veamos que podemos contruir una cota inferior para la log-verosimilitud:
Sea $Q(z)$ una distribucion sobre $z$ entonces podemos escribir:
$$
l(\theta)=\log p(x;\theta) = \log \sum\limits_zp(x,z ; \theta) = \log \sum\limits_zQ(z)\frac{p(x,z ; \theta)}{Q(z)}
$$

Ahora aplicando la Desigualdad de Jensen llegamos a:
$$
\log \sum\limits_zQ(z)\frac{p(x,z ; \theta)}{Q(z)} \ge \sum\limits_zQ(z)\log \frac{p(x,z ; \theta)}{Q(z)}
$$
Por lo tanto hemos encontrado una cota inferior de la log-verosimilitud valida para cualquier distribucion $Q$ sobre $z$. A esta cota la llamaremos **ELBO (Evidence Lower Bound)** visto como una función de x. Como queremos maximizar la verosimilitd lo mejor sería escoger una distribucion que hiciese que la igualdad se de. Por el teorema de antes sabemos que esto lo conseguimos si $\frac{p(x,z ; \theta)}{Q(z)}$ es una constante (estamos usando que el logaritmo es estrictamente convexo). Y eso a su vez lo tenemos si $Q(z) \propto p(x,z ; \theta)$. Haciendo manipulaciones sencillas usando que $Q$ es distribucion y probabilidad condicionada llegamos a que esto es lo mismo que: $Q(z) = p(z|x;\theta)$. Es decir, la probabilidad a posteriori de $z$.

Resumiendo todo este apartado, lo que hemos hecho es ver que podemos encontrar una cota para la log-verosimilitud y que la cota es óptima si elegimos que la distribución de $Q$ sea la distribución a posteriori de $z$.

### **2. Esquema EM**

El algoritmo EM sera un algoritmo iterativo de dos pasos:

(a) Determinamos la distribucion de $z$ $Q$ con los parámetros fijos. 

(b) Con la distribución $Q$ fija seleccionamos los parametros de forma que sean los que maximizan ELBO


Debemos notar que EM es un esquema y no un algoritmo concreto. Para especificarlo necesitamos concretar la familia parametrica a la que pertenece $p(x, z)$ normalmente concretando las familias de $p(x)$ y $p(z|x)$. 

Si por ejemplo asumimos $X \sim \mathcal{N}(\mu, \Sigma)$ y $Z|X \sim \operatorname{Multinomial}_n(\phi)$ entonces el algortimo es Mixture of Gaussians que desarrollaremos e implementaremos a continuación. Este ejemplo da sentido tambien a las variables latentes como adelantabamos al principio. Podemos interpretar que existen $n$ clusters y la muestra se genera primero eligiendo uno de los clusters y posteriormente de acuerdo a la normal asociada al cluster. 